# Apply a Trained Model Across an Atlas

The same pattern can be used with scikit-learn estimators, PyTorch models,
custom centroids, or other methods that accept dense expression minibatches.

By the end of this tutorial, you will be able to:

- define the cells, genes, and expression representation used for inference;
- apply a trained model in a single pass over the selected cells;
- associate each prediction with the correct Atlas cell;
- write compact prediction outputs back to `obs`;
- store probabilities or other multidimensional outputs separately.

## Before You Begin

This tutorial assumes that:

- preprocessing has already been completed;
- the model has already been trained;
- the model is available in the current Python session or has been loaded from
  a saved model file;
- the genes and expression representation used during training are known.

Objects such as scikit-learn estimators and PyTorch models are not stored
automatically in the `.sasql` Atlas. Save and load them separately using the
mechanism recommended by the corresponding framework.

Open the same Atlas database used throughout the advanced tutorials:



In [ ]:
import scatlaspy as sap

atlas = sap.Atlas(
    "./tmp/tutorials/basic_pbmc3k/pbmc3k_basic_copy.sasql",
    db_memory_limit="8GB",
)


## 1. Define the Inference Population

Build a read index that defines the cells, genes, and expression field supplied
to the model:



In [ ]:
atlas.build_read_index(
    cell_condition="filter_cells",
    gene_condition="filter_genes",
    use_hvg=True,
    use_data="data_scale",
)



In this example, inference is performed on:

- cells selected by `filter_cells`;
- genes selected by `filter_genes`;
- genes marked as highly variable;
- scaled expression values stored in `data_scale`.

```{important}
The current read index defines the inference population. If it selects filtered
cells, the model is applied to all selected cells rather than every cell stored
in the Atlas database.
```

## 2. Verify Model Compatibility

The model input must match the data representation defined by the read index.

Confirm that the model was trained using:

- the same genes;
- the same gene order;
- the same expression field;
- the same normalization, transformation, and scaling procedure;
- the same expected input data type.

Matching only the number of genes is not sufficient. Two matrices with the same
shape but different gene orders represent different model inputs.

For a scikit-learn estimator that exposes `n_features_in_`, you can perform a
basic dimensionality check:



In [ ]:
print(model.n_features_in_)



This check does not verify gene identity or order. The feature names and
preprocessing configuration used during training should be saved alongside the
model and compared with the current read index before inference.

## 3. Retrieve the Target Cell Order

A `single-pass` stream traverses the current read index once. Predictions must
be attached to cells in the same order.

Retrieve the corresponding cell identifiers:



In [ ]:
cell_index = atlas.query("""
    SELECT filter_cell_id, atlas_cell_id
    FROM obs
    WHERE filter_cell_id IS NOT NULL
    ORDER BY filter_cell_id
""")



The `filter_cell_id` column records the cell order used by the current read
index, while `atlas_cell_id` provides the persistent identifier used to join
results back to `obs`.

```{important}
This tutorial assumes that `get_minibatch_dense(pass_mode="single-pass")`
returns cells in ascending `filter_cell_id` order. Prediction-to-cell alignment
depends on this ordering guarantee.
```

## 4. Apply the Model in Minibatches

For compact one-dimensional outputs, such as class labels, predictions can be
written to a temporary database table as each minibatch is processed.

The example below assumes that `model.predict()` returns one label per cell:



In [ ]:
import numpy as np
import pandas as pd

conn = atlas.connection

atlas.execute_sql("""
    DROP TABLE IF EXISTS model_prediction_staging
""")

atlas.execute_sql("""
    CREATE TEMP TABLE model_prediction_staging (
        atlas_cell_id BIGINT,
        model_prediction VARCHAR
    )
""")

offset = 0

for X_batch in atlas.get_minibatch_dense(
    pass_mode="single-pass",
    batch_size=4096,
):
    predictions = np.asarray(model.predict(X_batch))

    if predictions.ndim != 1:
        raise ValueError(
            "This example expects one prediction value per cell."
        )

    n_batch = len(predictions)

    cell_batch = cell_index.iloc[offset : offset + n_batch].copy()

    if len(cell_batch) != n_batch:
        raise ValueError(
            "The prediction stream contains more cells than the read index."
        )

    prediction_batch = pd.DataFrame(
        {
            "atlas_cell_id": cell_batch["atlas_cell_id"].to_numpy(),
            "model_prediction": predictions.astype(str),
        }
    )

    conn.register("prediction_batch", prediction_batch)

    atlas.execute_sql("""
        INSERT INTO model_prediction_staging
        SELECT atlas_cell_id, model_prediction
        FROM prediction_batch
    """)

    conn.unregister("prediction_batch")

    offset += n_batch

if offset != len(cell_index):
    raise ValueError(
        "The prediction count does not match the number of cells "
        "in the current read index."
    )

print(f"Predicted {offset:,} cells")



This workflow keeps only one expression minibatch and one prediction minibatch
in memory at a time. The cell-index table remains much smaller than the full
cell-by-gene expression matrix.

Adjust `batch_size` according to the number of selected genes, the model, and
the available memory. Dense minibatches can still require substantial memory
when many genes are included.

## 5. Write Class Labels to obs

A one-dimensional output such as a class label can be stored as a column in
`obs`.

The following example creates or reuses a string column named
`model_prediction`:



In [ ]:
atlas.execute_sql("""
    ALTER TABLE obs
    ADD COLUMN IF NOT EXISTS model_prediction VARCHAR
""")



Clear the column before writing the current prediction run:



In [ ]:
atlas.execute_sql("""
    UPDATE obs
    SET model_prediction = NULL
""")



Then join the staged results to `obs` using `atlas_cell_id`:



In [ ]:
atlas.execute_sql("""
    UPDATE obs
    SET model_prediction = model_prediction_staging.model_prediction
    FROM model_prediction_staging
    WHERE obs.atlas_cell_id = model_prediction_staging.atlas_cell_id
""")



Remove the temporary staging table after the update:



In [ ]:
atlas.execute_sql("""
    DROP TABLE model_prediction_staging
""")



```{warning}
Clearing `model_prediction` removes results from any previous prediction run
stored in that column. Use a distinct column name when several models or model
versions must be retained.
```

Choose an SQL type that matches the model output:

| Output | Suggested storage |
|---|---|
| Text class label | `VARCHAR` column in `obs` |
| Integer class label | Integer column in `obs` |
| One continuous score | Floating-point column in `obs` |
| Multiple class probabilities | Separate result table |
| Embedding or latent representation | Separate matrix or result table |

## 6. Validate the Stored Predictions

Check how many selected cells received a prediction:



In [ ]:
atlas.query("""
    SELECT
        COUNT(*) AS selected_cells,
        COUNT(model_prediction) AS predicted_cells
    FROM obs
    WHERE filter_cell_id IS NOT NULL
""")



Inspect the distribution of predicted labels:



In [ ]:
atlas.query("""
    SELECT
        model_prediction,
        COUNT(*) AS n_cells
    FROM obs
    WHERE filter_cell_id IS NOT NULL
    GROUP BY model_prediction
    ORDER BY n_cells DESC
""")



Unexpected missing values may indicate:

- a mismatch between the read index and prediction stream;
- an interrupted inference run;
- an incorrect join key;
- model output with an unexpected shape.

## 7. Use a PyTorch Model

For PyTorch inference, place the model in evaluation mode and disable gradient
tracking:



In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = model.to(device)
model.eval()

with torch.inference_mode():
    for X_batch in atlas.get_minibatch_dense(
        pass_mode="single-pass",
        batch_size=4096,
    ):
        X_tensor = torch.as_tensor(
            X_batch,
            dtype=torch.float32,
            device=device,
        )

        logits = model(X_tensor)
        predictions = logits.argmax(dim=1).cpu().numpy()

        # Associate this prediction batch with the corresponding
        # atlas_cell_id values and write it to the staging table.



The cell alignment and database-writing steps are the same as in the
scikit-learn example.

GPU memory use depends on both `batch_size` and the model architecture. Reduce
the batch size when inference exceeds the available device memory.

## 8. Store Probabilities or Multidimensional Outputs

A probability matrix may contain one value for every cell and class:



In [ ]:
probabilities = model.predict_proba(X_batch)



Do not collect all probability batches with `np.vstack()` when the complete
matrix may be large. Write each batch incrementally to a dedicated result table
instead.

A probability table can include:

- `atlas_cell_id`;
- a model or prediction-run identifier;
- one probability column per class, or a class label and probability in
  long-table form.

Similarly, embeddings and latent representations should normally be stored in
a dedicated matrix-style result table rather than as many separate `obs`
columns.

Use `obs` for compact cell-level annotations and use separate result tables for
multidimensional outputs.

## Relationship to Built-in Tools

The general pattern used in this tutorial is:

1. define an analysis population through the read index;
2. fit or load a model using the same features and expression representation;
3. traverse the selected cells in a deterministic single pass;
4. apply the model to one minibatch at a time;
5. associate outputs with persistent cell identifiers;
6. store or export the results.

Built-in dimensionality-reduction and clustering tools may use related
fit-and-apply strategies internally. Custom methods can use the same Atlas
data-access interfaces without requiring the complete expression matrix to be
materialized in memory.

## Next Steps

Use {doc}`query-atlas-with-sql` to summarize predictions by sample, cluster,
condition, or existing annotation.

For examples of fitting models with the same minibatch interfaces, see:

- {doc}`train-logistic-regression-with-minibatches`;
- {doc}`train-pytorch-model-with-minibatches`;
- {doc}`implement-minibatch-kmeans`.